# Step 10: train model 7 (~30M) on FineWeb-Edu + TinyStories

Before starting: **Runtime → Change runtime type → a GPU** (one with bf16 support, e.g. L4 or A100), and upload `step10_bundle.zip` to Google Drive in a folder `mini-gpt`. Make the bundle from the repo folder:

`zip step10_bundle.zip step07_bpe_tokenizer.py step08_tinystories_tokenizer.py step10_fineweb_check.py step10a_tokenizer.py step10b_data.py step10_model.py step10_train.py data/fineweb/tokenizer_8k.json`

Run the cells in order. If Colab disconnects, run cells 1–4 again, then cell 6: training continues from the last checkpoint.

### 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 2. Connect Google Drive (checkpoints are saved there, so they survive a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. Unpack the code

In [ ]:
!mkdir -p /content/mini-gpt && unzip -o -q /content/drive/MyDrive/mini-gpt/step10_bundle.zip -d /content/mini-gpt
%cd /content/mini-gpt
!ls

### 4. Prepare the data (~7 GB download + encoding)
Skips the download if the files are already there.

In [ ]:
!python step10b_data.py --out /content/data

### 5. Speed test (30 steps): prints speed and the full-run time
If it says out of memory: add `--micro 32` here and in cell 6.

In [ ]:
!python step10_train.py --data /content/data --out /content/speedtest --speed_test

### 6. Full training. Keep this tab open.
Checkpoint every 500 steps and best model go to Drive: `MyDrive/mini-gpt/gpt_fineweb/`

In [ ]:
!python step10_train.py --data /content/data --out /content/drive/MyDrive/mini-gpt/gpt_fineweb
from google.colab import runtime
runtime.unassign()   # shut the runtime down when training ends (no idle cost)

### 7. When finished
Download `best.pt`, `history.json` and `train.log` from `MyDrive/mini-gpt/gpt_fineweb/` into `results/gpt_fineweb/` on your computer.
Then **Runtime → Disconnect and delete runtime**, so no more units are used.